# Stationary incompressible linear control problems
In this notebook, we describe how to solve linear control problems, when incompressibility constraints are imposed on the state variable.

## A stationary incompressible Stokes control problem
As an example, we consider the solution of the following stationary incompressible Stokes problem:

$$
\min_{\vec{v},\vec{u}} \frac{1}{2} \| \vec{v} - \vec{v}_d \| ^2 + \frac{\beta}{2} \| \vec{u} \| ^2
$$

subject to:

$$
    - \nabla^2 \; \vec{v} + \nabla p = \vec{u} + \vec{f} \qquad \mathrm{in} \; \Omega,
$$
$$
    - \nabla \cdot \vec{v} = 0 \qquad \mathrm{in} \; \Omega,
$$
$$
    \vec{v} = \vec{g} \qquad \mathrm{on} \; \partial \Omega.
$$

We consider the lid-driven cavity problem in $\Omega = (-1, 1)^2$, with force function $\vec{f}=[0, 0]^\top$ and boundary conditions given by

$$
    \vec{g} = [1, 0]^\top \quad \mathrm{on} \; \partial \Omega_1:=(-1,1) \times \{ 1\},
$$
$$
    \vec{g} = [0, 0]^\top \quad \mathrm{on} \; \partial \Omega \setminus \partial \Omega_1.
$$

We seek $\vec{v}_d = [0, 0]^\top$ as the desired state, and set $\beta = 10^{-3}$.

As a first task, we import all the important modules. Note that the linear solver requires the user to provide as an input the nullspace of the corresponding forward stationary Stokes problem. For this reason, we have to import from the class preconditioner the module ConstantNullspace, as we are solving for an enclosed flow.

In [ ]:
from firedrake import *
from control.preconditioner import ConstantNullspace
from control.control import Stationary

We can now build the mesh, define the finite element spaces from which we seek the solutions together with the boundary conditions for the velocity, the force function, and the desired state. We employ $\mathbf{P}_2$–$\mathbf{P}_1$ finite element pair. We mention that the pressure space can be passed directly to the linear solver.

In [ ]:
mesh = RectangleMesh(10, 10, 1.0, 1.0, originX=-1.0, originY=-1.0)

space_v = VectorFunctionSpace(mesh, "Lagrange", 2)
space_p = FunctionSpace(mesh, "Lagrange", 1)

# the boundary conditions
bcs_v = [DirichletBC(space_v, Constant((1.0, 0.0)), (4,)),
         DirichletBC(space_v, 0.0, (1, 2, 3))]


# the forward form
def forw_diff_operator(trial, test, u):
    # spatial differential operator for the forward problem
    return inner(grad(trial), grad(test)) * dx


# the desired state
def desired_state(test):
    space = test.function_space()

    # desired state
    v_d = Function(space, name="v_d")
    v_d.zero()

    return inner(v_d, test) * dx, v_d


# the force function
def force_f(test):
    space = test.function_space()

    # force function
    f = Function(space)
    f.zero()

    return inner(f, test) * dx


stationary_Stokes_control = Stationary(
    space_v, forw_diff_operator, desired_state=desired_state,
    force_function=force_f, space_p=space_p, bcs_v=bcs_v)

In order to solve the problem, we have now to call the module incompressible_linear_solve(). The call requires as an input the nullspace of the corresponding forward Stokes problem we are considering here. We employ the in-built preconditioned iterative method, that is running FGMRES for 50 iterations restarted every 10 iterations, applying as the preconditioner a block-triangular matrix with the (1,1)-block approximately inverted with 5 preconditioned GMRES iterations, and a Schur complement approximation based on the block-pressure convection–diffusion preconditioner; the preconditioner for the inner GMRES solver is based on the matching strategy. The default options for the vector- and pressure-mass matrices is a Jacobi iteration, while the pressure-stiffness matrix is approximated inexactly with 1 cycle of the hypre multigrid routine. Note that the options can be modified by passing to the call the kwarg auxiliary_sp. For example, suppose we want to run the inner solver for 3 iterations stopping once the relative residual has been reduced of one order of magnitude, apply 20 Chebyshev semi-iterations to approximately invert the vector- and pressure-mass matrices, and apply 2 cycle of hypre as an approximation of the inverse of the pressure-stiffness matrix.

In [ ]:
# option for the (1,1)-block solve
inner_solver_parameters = {
    "preconditioner": True,
    "linear_solver": "gmres",
    "maximum_iterations": 5,
    "relative_tolerance": 1.0e-2,
    "absolute_tolerance": 0.0,
    "monitor_convergence": False}

# employing Chebyshev for the (1,1)-block
e_min_v = 0.3924
e_max_v = 2.0598
sp_11block = {
    "ksp_type": "chebyshev",
    "pc_type": "jacobi",
    "ksp_chebyshev_eigenvalues": f"{e_min_v:.16e}, {e_max_v:.16e}",
    "ksp_chebyshev_esteig": "0.0,0.0,0.0,0.0",
    "ksp_chebyshev_esteig_steps": 0,
    "ksp_chebyshev_esteig_noisy": False,
    "ksp_max_it": 20,
    "ksp_atol": 0.0,
    "ksp_rtol": 0.0}

# employing Chebyshev for the pressure-mass matrix
e_min_p = 0.5
e_max_p = 2.0
sp_M_p = {
    "ksp_type": "chebyshev",
    "pc_type": "jacobi",
    "ksp_chebyshev_eigenvalues": f"{e_min_p:.16e}, {e_max_p:.16e}",
    "ksp_chebyshev_esteig": "0.0,0.0,0.0,0.0",
    "ksp_chebyshev_esteig_steps": 0,
    "ksp_chebyshev_esteig_noisy": False,
    "ksp_max_it": 20,
    "ksp_atol": 0.0,
    "ksp_rtol": 0.0}

# applying 2 cycles of the hypre multigrid
sp_K_p = {"ksp_type": "preonly",
          "pc_type": "hypre",
          "pc_hypre_type": "boomeramg",
          "ksp_max_it": 1,
          "pc_hypre_boomeramg_max_iter": 2,
          "ksp_atol": 0.0,
          "ksp_rtol": 0.0}

auxiliary_sp = {
    "sp_inner": inner_solver_parameters,
    "sp_11block": sp_11block,
    "sp_M_p": sp_M_p,
    "sp_K_p": sp_K_p}

stationary_Stokes_control.incompressible_linear_solve(
    ConstantNullspace(), auxiliary_sp=auxiliary_sp)